# Day 02: The Unscented Kalman Filter (UKF) - Nonlinear Manifolds & Sigma-Point Filtering
**State Estimation and Localization for Self-Driving Cars**

### 🎯 Learning Objectives:
1. Understand the fundamental limitation of Taylor series linearization in highly curved manifolds.
2. Master the **Scaled Unscented Transform (SUT)** and parameter selection ($\alpha, \beta, \kappa$).
3. Implement the generic multi-dimensional `UnscentedKalmanFilter` class via robust Cholesky matrix decomposition and circular angle mean unwrapping.
4. Benchmark UKF vs. EKF on high-rate turning maneuvers with severe polar bearing nonlinearities.
5. Visualize covariance confidence bounds and residual convergence via interactive Plotly dashboards.

---
## 1. Environment Setup

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
print('Environment ready: NumPy and Plotly loaded.')

---
## 2. Mathematical Foundation: The Scaled Unscented Transform (SUT)

The UKF replaces analytical Jacobian approximations with **deterministic sampling**:

$$\lambda = \alpha^2(L + \kappa) - L, \quad \gamma = \sqrt{L + \lambda}$$

$$\begin{aligned}
W_m^{(0)} &= \frac{\lambda}{L + \lambda}, \quad &W_c^{(0)} &= \frac{\lambda}{L + \lambda} + (1 - \alpha^2 + \beta) \\[4pt]
W_m^{(i)} &= \frac{1}{2(L + \lambda)}, \quad &W_c^{(i)} &= \frac{1}{2(L + \lambda)} \quad (i = 1, \dots, 2L)
\end{aligned}$$

Sigma Points via Cholesky decomposition $\mathbf{L}\mathbf{L}^T = \mathbf{P}$:
$$\boldsymbol{\mathcal{X}}^{(0)} = \boldsymbol{\mu}, \quad \boldsymbol{\mathcal{X}}^{(i)} = \boldsymbol{\mu} + \gamma \operatorname{col}_i(\mathbf{L}), \quad \boldsymbol{\mathcal{X}}^{(i+L)} = \boldsymbol{\mu} - \gamma \operatorname{col}_i(\mathbf{L})$$

### 🔁 Discrete UKF Recursive Algorithm:

1. **Time Update (Prediction)**:
   $$\boldsymbol{\mathcal{X}}_{k|k-1}^{(i)} = \mathbf{f}(\boldsymbol{\mathcal{X}}_{k-1}^{(i)}, \mathbf{u}_{k-1})$$
   $$\check{\mathbf{x}}_k = \sum_{i=0}^{2L} W_m^{(i)} \boldsymbol{\mathcal{X}}_{k|k-1}^{(i)}, \quad \check{\mathbf{P}}_k = \sum_{i=0}^{2L} W_c^{(i)} (\boldsymbol{\mathcal{X}}_{k|k-1}^{(i)} - \check{\mathbf{x}}_k)(\boldsymbol{\mathcal{X}}_{k|k-1}^{(i)} - \check{\mathbf{x}}_k)^T + \mathbf{Q}_{k-1}$$

2. **Measurement Update (Correction)**:
   $$\boldsymbol{\mathcal{Y}}_{k|k-1}^{(i)} = \mathbf{h}(\boldsymbol{\mathcal{X}}_{k|k-1}^{(i)})$$
   $$\check{\mathbf{y}}_k = \sum_{i=0}^{2L} W_m^{(i)} \boldsymbol{\mathcal{Y}}_{k|k-1}^{(i)}, \quad \mathbf{S}_k = \sum_{i=0}^{2L} W_c^{(i)} (\boldsymbol{\mathcal{Y}}_{k|k-1}^{(i)} - \check{\mathbf{y}}_k)(\boldsymbol{\mathcal{Y}}_{k|k-1}^{(i)} - \check{\mathbf{y}}_k)^T + \mathbf{R}_k$$
   $$\mathbf{P}_{xy, k} = \sum_{i=0}^{2L} W_c^{(i)} (\boldsymbol{\mathcal{X}}_{k|k-1}^{(i)} - \check{\mathbf{x}}_k)(\boldsymbol{\mathcal{Y}}_{k|k-1}^{(i)} - \check{\mathbf{y}}_k)^T$$
   $$\mathbf{K}_k = \mathbf{P}_{xy, k}\mathbf{S}_k^{-1}, \quad \hat{\mathbf{x}}_k = \check{\mathbf{x}}_k + \mathbf{K}_k(\mathbf{y}_k - \check{\mathbf{y}}_k), \quad \hat{\mathbf{P}}_k = \check{\mathbf{P}}_k - \mathbf{K}_k \mathbf{S}_k \mathbf{K}_k^T$$

---
## 3. Implementing the `UnscentedKalmanFilter` Class

### 📝 Exercise 1: Implement the Generic UKF Class

In [ ]:
def wrap_angle(angle):
    """Wraps angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class UnscentedKalmanFilter:
    """Unscented Kalman Filter (UKF) with Scaled Unscented Transform."""
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray, alpha=0.5, beta=2.0, kappa=0.0):
        self.x = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.L = self.x.shape[0]
        self.alpha, self.beta, self.kappa = alpha, beta, kappa
        self.lambda_ = self.alpha**2 * (self.L + self.kappa) - self.L
        self.gamma = np.sqrt(self.L + self.lambda_)
        self.num_sigmas = 2 * self.L + 1
        self.Wm = np.zeros(self.num_sigmas)
        self.Wc = np.zeros(self.num_sigmas)
        self.Wm[0] = self.lambda_ / (self.L + self.lambda_)
        self.Wc[0] = self.Wm[0] + (1.0 - self.alpha**2 + self.beta)
        for i in range(1, self.num_sigmas):
            self.Wm[i] = 1.0 / (2.0 * (self.L + self.lambda_))
            self.Wc[i] = self.Wm[i]
            
        self.latest_innovation = None
        self.latest_innovation_cov = None
        self.latest_gain = None
            
    def generate_sigma_points(self, x_mean: np.ndarray, P_cov: np.ndarray, angle_indices: list = None):
        # -------------------------------------------------------------------------
        # TODO 1.1: Implement Cholesky factorization and 2L+1 sigma points generation
        # L_mat = chol(P_cov)
        # sigmas[:, 0] = x_mean, sigmas[:, i+1] = x_mean + gamma*L_i, sigmas[:, i+1+L] = x_mean - gamma*L_i
        # -------------------------------------------------------------------------
        pass  # <-- YOUR CODE HERE
        
    def predict(self, f_func, Q: np.ndarray, u: np.ndarray = None, angle_indices: list = None):
        # -------------------------------------------------------------------------
        # TODO 1.2: Implement UKF prediction step
        # 1. Transform sigma points through nonlinear dynamics: sigmas_pred = f(sigmas, u)
        # 2. Recombine mean: x_pred = sum(Wm_i * sigmas_pred_i) (use angle unwrapping for angle_indices)
        # 3. Recombine covariance: P_pred = sum(Wc_i * (diff @ diff.T)) + Q
        # -------------------------------------------------------------------------
        pass  # <-- YOUR CODE HERE
        
    def update(self, y: np.ndarray, h_func, R: np.ndarray, 
               meas_angle_indices: list = None, state_angle_indices: list = None):
        # -------------------------------------------------------------------------
        # TODO 1.3: Implement UKF measurement update
        # 1. Project sigma points to measurement space: gamma_meas = h(sigmas)
        # 2. Compute predicted measurement mean y_pred (with angle unwrapping) and covariance Py
        # 3. Compute cross-covariance Pxy
        # 4. Compute Kalman gain K = Pxy * inv(Py), correct state and covariance
        # -------------------------------------------------------------------------
        pass  # <-- YOUR CODE HERE


In [ ]:
def wrap_angle(angle):
    """Wraps angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class UnscentedKalmanFilter:
    """Generic Multi-Dimensional Unscented Kalman Filter (UKF) with Scaled Unscented Transform.
    
    Completely derivative-free state estimation supporting arbitrary dimension L, nonlinear
    transition functions f(x, u), observation models h(x), and robust angle unwrapping.
    """
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray, alpha=0.5, beta=2.0, kappa=0.0):
        self.x = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.L = self.x.shape[0]
        
        self.alpha, self.beta, self.kappa = alpha, beta, kappa
        self.lambda_ = self.alpha**2 * (self.L + self.kappa) - self.L
        self.gamma = np.sqrt(self.L + self.lambda_)
        self.num_sigmas = 2 * self.L + 1
        
        # Merwe Scaled Weights
        self.Wm = np.zeros(self.num_sigmas)
        self.Wc = np.zeros(self.num_sigmas)
        self.Wm[0] = self.lambda_ / (self.L + self.lambda_)
        self.Wc[0] = self.Wm[0] + (1.0 - self.alpha**2 + self.beta)
        for i in range(1, self.num_sigmas):
            self.Wm[i] = 1.0 / (2.0 * (self.L + self.lambda_))
            self.Wc[i] = self.Wm[i]
            
        self.latest_innovation = None
        self.latest_innovation_cov = None
        self.latest_gain = None
            
    def generate_sigma_points(self, x_mean: np.ndarray, P_cov: np.ndarray, angle_indices: list = None):
        """Generates 2L+1 sigma points via robust Cholesky decomposition L = chol(P)."""
        P_sym = 0.5 * (P_cov + P_cov.T)
        try:
            L_mat = np.linalg.cholesky(P_sym)
        except np.linalg.LinAlgError:
            eigvals, eigvecs = np.linalg.eigh(P_sym)
            eigvals = np.maximum(eigvals, 1e-9)
            P_sym = eigvecs @ np.diag(eigvals) @ eigvecs.T
            L_mat = np.linalg.cholesky(P_sym)
            
        sigmas = np.zeros((self.L, self.num_sigmas))
        sigmas[:, 0] = x_mean.flatten()
        for i in range(self.L):
            col = L_mat[:, i]
            sigmas[:, i + 1] = x_mean.flatten() + self.gamma * col
            sigmas[:, i + 1 + self.L] = x_mean.flatten() - self.gamma * col
        if angle_indices is not None:
            for idx in angle_indices:
                sigmas[idx, :] = wrap_angle(sigmas[idx, :])
        return sigmas
        
    def predict(self, f_func, Q: np.ndarray, u: np.ndarray = None, angle_indices: list = None):
        """Executes the UKF Sigma-Point State and Covariance Prediction Step."""
        sigmas = self.generate_sigma_points(self.x, self.P, angle_indices=angle_indices)
        sigmas_pred = np.zeros_like(sigmas)
        for i in range(self.num_sigmas):
            sp = sigmas[:, i:i+1]
            if u is not None:
                sigmas_pred[:, i:i+1] = f_func(sp, u).reshape(-1, 1)
            else:
                sigmas_pred[:, i:i+1] = f_func(sp).reshape(-1, 1)
            
        # Recombine mean with circular angle unwrapping
        x_pred = np.zeros((self.L, 1))
        for row in range(self.L):
            if angle_indices is not None and row in angle_indices:
                ref = sigmas_pred[row, 0]
                unwrapped = ref + wrap_angle(sigmas_pred[row, :] - ref)
                x_pred[row, 0] = wrap_angle(np.sum(self.Wm * unwrapped))
            else:
                x_pred[row, 0] = np.sum(self.Wm * sigmas_pred[row, :])
                
        P_pred = np.zeros((self.L, self.L))
        for i in range(self.num_sigmas):
            diff = sigmas_pred[:, i:i+1] - x_pred
            if angle_indices is not None:
                for idx in angle_indices:
                    diff[idx, 0] = wrap_angle(diff[idx, 0])
            P_pred += self.Wc[i] * (diff @ diff.T)
        P_pred += Q
        
        self.x = x_pred
        self.P = 0.5 * (P_pred + P_pred.T)
        return self.x.copy(), self.P.copy()
        
    def update(self, y: np.ndarray, h_func, R: np.ndarray, 
               meas_angle_indices: list = None, state_angle_indices: list = None):
        """Executes the UKF Measurement Correction Step."""
        y_vec = np.asarray(y, dtype=np.float64).reshape(-1, 1)
        m = y_vec.shape[0]
        sigmas = self.generate_sigma_points(self.x, self.P, angle_indices=state_angle_indices)
        gamma_meas = np.zeros((m, self.num_sigmas))
        for i in range(self.num_sigmas):
            gamma_meas[:, i:i+1] = h_func(sigmas[:, i:i+1]).reshape(-1, 1)
            
        # Recombine measurement mean with circular angle unwrapping
        y_pred = np.zeros((m, 1))
        for row in range(m):
            if meas_angle_indices is not None and row in meas_angle_indices:
                ref = gamma_meas[row, 0]
                unwrapped = ref + wrap_angle(gamma_meas[row, :] - ref)
                y_pred[row, 0] = wrap_angle(np.sum(self.Wm * unwrapped))
            else:
                y_pred[row, 0] = np.sum(self.Wm * gamma_meas[row, :])
                
        Py = np.zeros((m, m))
        Pxy = np.zeros((self.L, m))
        for i in range(self.num_sigmas):
            dy = gamma_meas[:, i:i+1] - y_pred
            if meas_angle_indices is not None:
                for idx in meas_angle_indices:
                    dy[idx, 0] = wrap_angle(dy[idx, 0])
            dx = sigmas[:, i:i+1] - self.x
            if state_angle_indices is not None:
                for idx in state_angle_indices:
                    dx[idx, 0] = wrap_angle(dx[idx, 0])
            Py += self.Wc[i] * (dy @ dy.T)
            Pxy += self.Wc[i] * (dx @ dy.T)
        Py += R
        
        K = Pxy @ np.linalg.inv(Py)
        residual = y_vec - y_pred
        if meas_angle_indices is not None:
            for idx in meas_angle_indices:
                residual[idx, 0] = wrap_angle(residual[idx, 0])
                
        self.x = self.x + K @ residual
        if state_angle_indices is not None:
            for idx in state_angle_indices:
                self.x[idx, 0] = wrap_angle(self.x[idx, 0])
                
        self.P = self.P - K @ Py @ K.T
        self.P = 0.5 * (self.P + self.P.T)
        
        self.latest_innovation = residual
        self.latest_innovation_cov = Py
        self.latest_gain = K
        
        return self.x.copy(), self.P.copy()

print('Generic UnscentedKalmanFilter class compiled successfully.')

---
## 4. Analytical Linearization (EKF) vs. Derivative-Free Sampling (UKF)

To benchmark UKF against EKF under aggressive nonlinear vehicle maneuvers:

### 📐 The Classical EKF Partial Derivatives:
Recall that the baseline EKF relies on locally evaluating the Jacobian matrices $\mathbf{F} = \frac{\partial \mathbf{f}}{\partial \mathbf{x}}$ and $\mathbf{H} = \frac{\partial \mathbf{h}}{\partial \mathbf{x}}$:
$$\mathbf{F} = \begin{bmatrix} 1 & 0 & \cos(\theta)\Delta t & -v \sin(\theta)\Delta t \\ 0 & 1 & \sin(\theta)\Delta t & v \cos(\theta)\Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \quad
\mathbf{H} = \begin{bmatrix} \frac{p_x}{r} & \frac{p_y}{r} & 0 & 0 \\[6pt] -\frac{p_y}{r^2} & \frac{p_x}{r^2} & 0 & 0 \end{bmatrix}, \quad r = \sqrt{p_x^2 + p_y^2}$$

**Why EKF degrades under severe angular rates and bearing noise:**
- EKF truncates the Taylor expansion at $\mathcal{O}(\Delta \mathbf{x}^1)$, ignoring curvature terms $\frac{1}{2}\Delta\mathbf{x}^T \nabla^2 \mathbf{f}\,\Delta\mathbf{x}$. When $\sigma_\phi$ is large or yaw rates are high, the prior Gaussian is severely distorted into a non-Gaussian banana-shaped manifold.
- In contrast, the **UKF** propagates $2L+1 = 9$ deterministic sigma points through the exact nonlinear equations $\mathbf{f}(\cdot)$ and $\mathbf{h}(\cdot)$, capturing posterior mean and covariance with 2nd-order Taylor accuracy without evaluating any matrix of partial derivatives.

---
## 5. Benchmark Challenge: EKF vs. UKF on Aggressive Nonlinear Maneuver

In [ ]:
dt = 0.1
T_total = 40.0
N_steps = int(T_total / dt)
time = np.linspace(0, T_total, N_steps)

x_true = np.array([10.0, 5.0, 12.0, 0.0]).reshape(4, 1)
x_true_all = np.zeros((4, N_steps))

Q = np.diag([0.05**2, 0.05**2, 0.1**2, 0.03**2])
R = np.diag([0.8**2, np.deg2rad(3.0)**2])

def motion_model(x, u):
    px, py, v, theta = x.flatten()
    a, omega = u.flatten()
    px_next = px + v * np.cos(theta) * dt
    py_next = py + v * np.sin(theta) * dt
    v_next = v + a * dt
    theta_next = wrap_angle(theta + omega * dt)
    return np.array([px_next, py_next, v_next, theta_next]).reshape(4, 1)

def measurement_model(x):
    px, py = x[0, 0], x[1, 0]
    r = np.sqrt(px**2 + py**2)
    phi = np.arctan2(py, px)
    return np.array([r, phi]).reshape(2, 1)

def get_F_jacobian(x):
    px, py, v, theta = x.flatten()
    return np.array([
        [1.0, 0.0, np.cos(theta) * dt, -v * np.sin(theta) * dt],
        [0.0, 1.0, np.sin(theta) * dt,  v * np.cos(theta) * dt],
        [0.0, 0.0, 1.0,                 0.0],
        [0.0, 0.0, 0.0,                 1.0]
    ])

def get_H_jacobian(x):
    px, py = x[0, 0], x[1, 0]
    r2 = max(px**2 + py**2, 1e-6)
    r = np.sqrt(r2)
    return np.array([
        [px / r,       py / r,       0.0, 0.0],
        [-py / r2,     px / r2,      0.0, 0.0]
    ])

measurements = []
for k in range(N_steps):
    t = time[k]
    a = 0.5 * np.cos(0.2 * t)
    omega = 0.25 * np.sin(0.15 * t)
    u = np.array([a, omega]).reshape(2, 1)
    w = np.random.multivariate_normal(np.zeros(4), Q).reshape(4, 1)
    x_true = motion_model(x_true, u) + w
    x_true[3, 0] = wrap_angle(x_true[3, 0])
    x_true_all[:, k] = x_true.flatten()
    v_noise = np.random.multivariate_normal(np.zeros(2), R).reshape(2, 1)
    y = measurement_model(x_true) + v_noise
    y[1, 0] = wrap_angle(y[1, 0])
    measurements.append((u, y))

x0_init = np.array([8.0, 3.0, 10.0, np.deg2rad(10.0)]).reshape(4, 1)
P0_init = np.diag([5.0**2, 5.0**2, 5.0**2, np.deg2rad(20.0)**2])

x_ekf = x0_init.copy()
P_ekf = P0_init.copy()
x_ekf_all = np.zeros((4, N_steps))

ukf = UnscentedKalmanFilter(x0_init, P0_init, alpha=0.5, beta=2.0, kappa=0.0)
x_ukf_all = np.zeros((4, N_steps))

for k in range(N_steps):
    u, y = measurements[k]
    # EKF Execution
    F = get_F_jacobian(x_ekf)
    x_ekf = motion_model(x_ekf, u)
    P_ekf = F @ P_ekf @ F.T + Q
    H = get_H_jacobian(x_ekf)
    y_pred_ekf = measurement_model(x_ekf)
    res_ekf = y - y_pred_ekf
    res_ekf[1, 0] = wrap_angle(res_ekf[1, 0])
    S_ekf = H @ P_ekf @ H.T + R
    K_ekf = P_ekf @ H.T @ np.linalg.inv(S_ekf)
    x_ekf = x_ekf + K_ekf @ res_ekf
    x_ekf[3, 0] = wrap_angle(x_ekf[3, 0])
    P_ekf = (np.eye(4) - K_ekf @ H) @ P_ekf
    x_ekf_all[:, k] = x_ekf.flatten()

    # Generic UKF Execution
    ukf.predict(motion_model, Q, u=u, angle_indices=[3])
    ukf.update(y, measurement_model, R, meas_angle_indices=[1], state_angle_indices=[3])
    x_ukf_all[:, k] = ukf.x.flatten()

rmse_ekf_pos = np.sqrt(np.mean((x_true_all[0, :] - x_ekf_all[0, :])**2 + (x_true_all[1, :] - x_ekf_all[1, :])**2))
rmse_ukf_pos = np.sqrt(np.mean((x_true_all[0, :] - x_ukf_all[0, :])**2 + (x_true_all[1, :] - x_ukf_all[1, :])**2))

print(f'Position RMSE -> EKF: {rmse_ekf_pos:.3f} m | UKF: {rmse_ukf_pos:.3f} m')

---
## 6. Comparative Visualizations: UKF vs. EKF via Plotly

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Trajectory Tracking: EKF vs. UKF (EKF: {rmse_ekf_pos:.2f}m, UKF: {rmse_ukf_pos:.2f}m)',
        'Position Error Convergence Over Time'
    )
)

fig.add_trace(go.Scatter(x=x_true_all[0, :], y=x_true_all[1, :], mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_ekf_all[0, :], y=x_ekf_all[1, :], mode='lines', name=f'EKF (RMSE: {rmse_ekf_pos:.2f}m)', line=dict(color='red', dash='dash', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_ukf_all[0, :], y=x_ukf_all[1, :], mode='lines', name=f'UKF (RMSE: {rmse_ukf_pos:.2f}m)', line=dict(color='blue', width=2)), row=1, col=1)

err_ekf = np.sqrt((x_true_all[0, :] - x_ekf_all[0, :])**2 + (x_true_all[1, :] - x_ekf_all[1, :])**2)
err_ukf = np.sqrt((x_true_all[0, :] - x_ukf_all[0, :])**2 + (x_true_all[1, :] - x_ukf_all[1, :])**2)
fig.add_trace(go.Scatter(x=time, y=err_ekf, mode='lines', name='EKF Error [m]', line=dict(color='red', dash='dash', width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=time, y=err_ukf, mode='lines', name='UKF Error [m]', line=dict(color='blue', width=1.5)), row=1, col=2)

fig.update_layout(title_text='Day 2 UKF vs. EKF Benchmark: Nonlinear Manifold Tracking', template='plotly_white', height=500, width=1100)
fig.show()